In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_csv("car_fuel_efficiency_2026.csv")
columns = ["engine_displacement", "horsepower", "vehicle_weight", "model_year", "fuel_efficiency_mpg"]
df = df[columns].copy()


## Questions 1 and 2


In [ ]:
df.isna().sum()
df.horsepower.median()


## Split and helper functions


In [ ]:
def split_data(data, seed):
    n = len(data)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    idx = np.arange(n)
    np.random.seed(seed)
    np.random.shuffle(idx)
    data = data.iloc[idx]
    df_train = data.iloc[n_val + n_test:].reset_index(drop=True)
    df_val = data.iloc[:n_val].reset_index(drop=True)
    df_test = data.iloc[n_val:n_val + n_test].reset_index(drop=True)
    return df_train, df_val, df_test

def prepare_X(data, fill_value):
    X = data.drop(columns=["fuel_efficiency_mpg"]).copy()
    X = X.fillna(fill_value)
    return X.values

def train_linear_regression(X, y, r=0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    w = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w[0], w[1:]

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())


## Question 3


In [ ]:
df_train, df_val, df_test = split_data(df, 42)
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
mean_hp = df_train.horsepower.mean()

for fill_value in [0, mean_hp]:
    X_train = prepare_X(df_train, fill_value)
    X_val = prepare_X(df_val, fill_value)
    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)
    print(fill_value, round(rmse(y_val, y_pred), 2))


## Question 4


In [ ]:
X_train = prepare_X(df_train, 0)
X_val = prepare_X(df_val, 0)

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression(X_train, y_train, r=r)
    y_pred = w0 + X_val.dot(w)
    print(r, round(rmse(y_val, y_pred), 2))


## Question 5


In [ ]:
scores = []

for seed in range(10):
    df_train, df_val, df_test = split_data(df, seed)
    X_train = prepare_X(df_train, 0)
    X_val = prepare_X(df_val, 0)
    y_train = df_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values
    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)
    scores.append(rmse(y_val, y_pred))

round(np.std(scores), 3)


## Question 6


In [ ]:
df_train, df_val, df_test = split_data(df, 9)
df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)

X_full_train = prepare_X(df_full_train, 0)
X_test = prepare_X(df_test, 0)
y_full_train = df_full_train.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

w0, w = train_linear_regression(X_full_train, y_full_train, r=0.001)
y_pred = w0 + X_test.dot(w)
round(rmse(y_test, y_pred), 3)


## Answers

1. `horsepower`
2. `204`
3. With mean
4. `0`
5. `0.029`
6. `2.236`
